In [1]:
import ast
import os

In [2]:
from urllib.parse import urlparse

def get_filename(url: str):
    if url[-4:] == ".git":
        url = url[:-4]
        
    parts = urlparse(url).path.strip("/").split("/")

    if len(parts) >= 2:
        username = parts[0]
        project_name = parts[1]
        result = f"{username}-{project_name}"
        return result

In [3]:
repo = "https://github.com/princ3kr/Notebook-LM-Mini"

filename = get_filename(repo)
dir = f"../temp/{filename}"

if os.path.isdir(dir):
    print("Directory already exists")

Directory already exists


In [4]:
import subprocess

if os.path.isdir(dir):
    print("Directory already exists")
else:
    try:
        subprocess.run(["git", "clone", repo, dir], check=True)
        print("Clone successful!")
    except subprocess.CalledProcessError as e:
        print(f"Git command failed with error: {e}")


Directory already exists


In [5]:
# tree = ast.parse(files["src\\services\\tutor_service.py"])
# print(ast.dump(tree, indent=2))

def parse_file(source_code):
    tree = ast.parse(source_code)
    import_modules, import_names, classes, functions = [], [], [], []
    
    for node in ast.walk(tree):
        if isinstance(node, ast.ImportFrom):
            import_modules.append(node.module)
            import_names.append([alias.name for alias in node.names])

        if isinstance(node, ast.ClassDef):
            classes.append({
                "name": node.name,
                "line_start": node.lineno,
                "line_end": node.end_lineno
            })
            
        if isinstance(node, ast.FunctionDef):
            functions.append({
                "name": node.name,
                "line_start": node.lineno,
                "line_end": node.end_lineno
            })
                
    return {"import_modules": import_modules, "import_names": import_names, "classes": classes, "functions": functions}

In [6]:
ignores = { ".git", ".gitignore", ".lock", ".venv", "__pycache__", "node_modules", ".vscode", "pyproject.toml", "__init__.py", ".python-version", "requirements.txt" }

def get_structure(directory: str):
    for root, dirs, files in os.walk(directory):
        dirs[:] = sorted([d for d in dirs if d not in ignores])

        rel_dir = os.path.relpath(root, directory)
        
        if rel_dir == ".":
            level = 0
            display_name = os.path.basename(os.path.abspath(directory))
        else:
            level = rel_dir.count(os.sep) + 1
            display_name = os.path.basename(root)

        indent = "│   " * (level - 1) if level > 0 else ""
        connector = "├── " if level > 0 else ""
        print(f"{indent}{connector}{display_name}/")

        file_indent = "│   " * level
        sorted_files = sorted(files)
        for i, name in enumerate(sorted_files):
            if name not in ignores or name == "README.md":
                file_connector = "└── " if (i == len(sorted_files) - 1 and not dirs) else "├── "
                print(f"{file_indent}{file_connector}{name}")


def get_files(directory: str):
    inventory = {}
    for root, dirs, files in os.walk(directory):
        rel_dir = os.path.relpath(root, directory)
        if rel_dir == ".":
            rel_dir = ""
        
        for name in files:
            if (name not in ignores) and (name == "README.md" or name.endswith(".py")):
                full_path = os.path.join(root, name)

                with open(full_path, "r", encoding="utf-8") as f:
                    content = f.read()

                    parsed_structure = {"import_modules": [], "import_names": [], "classes": [], "functions": []}

                    if name.endswith(".py"):
                        parsed_structure = parse_file(content)

                    parsed_structure['content'] = content

                    inventory[os.path.join(rel_dir, name)] = parsed_structure
                        

    return inventory

In [7]:
# get_structure(dir)
files = get_files(directory=dir)

In [8]:
for imports in files['src\services\diagnoser_service.py']:
    print(imports)

import_modules
import_names
classes
functions
content


<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:1: SyntaxWarning: invalid escape sequence '\s'
C:\Users\PRINCE KUMAR\AppData\Local\Temp\ipykernel_27188\1899480365.py:1: SyntaxWarning: invalid escape sequence '\s'
  for imports in files['src\services\diagnoser_service.py']:


In [9]:
import networkx as nx
from pyvis.network import Network
from neo4j import GraphDatabase
import json

class BuildGraph:
    def __init__(self, files: dict):
        self.G = nx.DiGraph()
        self.files = {filepath.replace("\\", "/"): data for filepath, data in files.items()}
        self.name_index = self.build_name_index()

    def build_name_index(self) -> dict:
        name_index = {}
        for filepath in self.files:
            for classname in self.files.get(filepath, {}).get("classes", []):
                name_index[classname['name']] = filepath
            for funcname in self.files.get(filepath, {}).get("functions", []):
                name_index[funcname['name']] = filepath
        return name_index

    def get_path(self) -> dict:
        cleared_files = {}
        for filepath in self.files:
            imports = []
            modules = self.files[filepath].get("import_modules", [])
            names = self.files[filepath].get("import_names", [])

            for imp, imp_names in zip(modules, names):
                if not imp or not imp.startswith("src."): ## harcoded!
                    continue

                target = imp.replace(".", "/") + ".py" ## hardcoded!
                if target in self.files:
                    imports.append(target)
                else:
                    for name in imp_names:
                        if name in self.name_index:
                            imports.append(self.name_index[name])
                            
            cleared_files[filepath] = imports
        return cleared_files

    def get_node_style(self, filepath):
        """Returns a more premium color palette and node properties."""
        if "services" in filepath:
            color = {"background": "#3b82f6", "border": "#60a5fa", "highlight": "#93c5fd"}
        elif "models" in filepath:
            color = {"background": "#10b981", "border": "#34d399", "highlight": "#6ee7b7"}
        elif "database" in filepath:
            color = {"background": "#f59e0b", "border": "#fbbf24", "highlight": "#fcd34d"}
        else:
            color = {"background": "#8b5cf6", "border": "#a78bfa", "highlight": "#c4b5fd"}
        
        return color

    def build(self):
        cleared_path = self.get_path()
        for source, targets in cleared_path.items():
            style = self.get_node_style(source)
            self.G.add_node(
                source, 
                label=source.split("/")[-1],
                title=source,
                color=style,
                shadow=True,
                shape="dot",
                size=25,
                font={"color": "white", "size": 14, "face": "Segoe UI, Tahoma, Geneva, Verdana, sans-serif"}
            )
            for target in targets:
                self.G.add_edge(source, target)

    def push_to_neo4j(self, uri, user, password):
        driver = GraphDatabase.driver(
            uri, auth=(user, password)
        )
        
        repo_id = filename

        with driver.session() as session:
            for node in self.G.nodes():
                imports = [
                    name for names in self.files[node]['import_names'] 
                    for name in names if name in self.name_index
                ]

                query = '''
                    MERGE (r:Repo {repo_id: $repo_id})
                    MERGE (f:File {path: $path, repo_id: $repo_id})
                    MERGE (r)-[:CONTAINS]->(f)
                    SET f.name = $name,
                        f.classes = $classes,
                        f.imports = $imports,
                        f.functions = $functions
                '''
                session.run(
                    query,
                    name=node.split('/')[-1],
                    path=node,
                    classes=[c['name'] for c in self.files.get(node, {}).get("classes", [])],
                    imports=imports,
                    functions=[f['name'] for f in self.files.get(node, {}).get("functions", [])],
                    repo_id=repo_id
                )
                
            for source, target in self.G.edges():
                query = '''
                    MATCH (a:File {path: $source, repo_id: $repo_id})
                    MATCH (b:File {path: $target, repo_id: $repo_id})
                    MERGE (a)-[:IMPORTS]->(b)
                '''
                session.run(query, source=source, target=target, repo_id=repo_id)
                

    def show(self, filename="graph.html"):
        net = Network(
            directed=True, 
            notebook=True, 
            cdn_resources='remote', 
            height="750px", 
            width="100%", 
            bgcolor="#111827",
            font_color="white"
        )
        
        net.from_nx(self.G)

        options = {
            "edges": {
                "color": {"inherit": "from", "opacity": 0.5},
                "smooth": {"type": "continuous", "roundness": 0.5},
                "arrows": {"to": {"enabled": True, "scaleFactor": 0.5}},
                "width": 1.5
            },
            "physics": {
                "forceAtlas2Based": {
                    "gravitationalConstant": -60,
                    "centralGravity": 0.005,
                    "springLength": 150,
                    "springStrength": 0.08,
                    "damping": 0.4,
                    "avoidOverlap": 0.5
                },
                "maxVelocity": 50,
                "minVelocity": 0.1,
                "solver": "forceAtlas2Based",
                "stabilization": {"enabled": True, "iterations": 1000}
            },
            "interaction": {
                "hover": True,
                "navigationButtons": True,
                "multiselect": True,
                "tooltipDelay": 200
            }
        }
        
        net.set_options(json.dumps(options))
        
        return net.write_html(filename)


In [10]:
builder = BuildGraph(files)
builder.build()
builder.show()

In [11]:
builder.push_to_neo4j("bolt://localhost:7687", "neo4j", "pk142145")

ServiceUnavailable: Couldn't connect to localhost:7687 (resolved to ('[::1]:7687', '127.0.0.1:7687')):
Failed to establish connection to ResolvedIPv6Address(('::1', 7687, 0, 0)) (reason [WinError 10061] No connection could be made because the target machine actively refused it)
Failed to establish connection to ResolvedIPv4Address(('127.0.0.1', 7687)) (reason [WinError 10061] No connection could be made because the target machine actively refused it)

In [12]:
import chromadb
from dotenv import load_dotenv
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

load_dotenv()
openai_key = os.getenv('OPENAI_API_KEY')

class VectorStore:
    def __init__(self, files:dict, path='../chromadb', collection_name:str='database'):
        self.files = files
        self.client = chromadb.PersistentClient(path=path)
        ef = OpenAIEmbeddingFunction(api_key=openai_key, model_name="text-embedding-3-small")
        self.collection = self.client.get_or_create_collection(name=collection_name, embedding_function=ef)

    def build(self, max_module_lines=100, overlap=10):
        self.chunks = []
        for file in self.files.keys():
            content = self.files[file]['content']
            lines = content.split('\n')
            classes = self.files[file]['classes']
            functions = self.files[file]['functions']

            for i in range(0, len(lines), max_module_lines):
                start = max(0, i - overlap)
                end = min(len(lines), i + overlap + max_module_lines)
                chunk_lines = lines[start : end]
                self.chunks.append([
                    "\n".join(chunk_lines),
                    {
                        "path": file,
                        "filename": file.split("/")[-1],
                        "function_name": "module_level_segment",
                        "class_name": "module_level",
                        "line_start": start + 1,
                        "line_end": end
                    }
                ])

            for func in functions:
                parent = 'module_level'
                for c in classes:
                    if func['line_start'] >= c['line_start'] and func['line_start'] <= c['line_end']:
                        parent = c['name']
                        break
                
                start = max(0, func['line_start'] - overlap)
                end = min(len(lines), func['line_end'] + overlap)
                chunk = "\n".join(lines[start:end])
                metadata = {
                    "path": file.replace("\\", "/"),
                    "filename": file.split("/")[-1],
                    "function_name": func['name'],
                    "class_name": parent,
                    "line_start": start + 1,
                    "line_end": end
                }
                self.chunks.append([chunk, metadata])

    def push(self):
        if not self.chunks:
            print("No chunks to push. Run build() first.")
            return

        documents = [c[0] for c in self.chunks]
        metadatas = [c[1] for c in self.chunks]

        ids = [f"{m['path']}:{m['function_name']}:{i}" for i, m in enumerate(metadatas)]

        self.collection.add(
            ids=ids,
            documents=documents,
            metadatas=metadatas
        )
        print(f"Successfully pushed {len(documents)} chunks to collection '{self.collection.name}'.")


    def search(self, query: str, top_k: int = 5):
        results = self.collection.query(
            query_texts=[query],
            n_results=top_k
        )
        return results

In [13]:
client = VectorStore(files, collection_name='prototype')
# client.build()

In [ ]:
# client.push()

Successfully pushed 92 chunks to collection 'prototype'.


In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import BaseModel, Field
from typing import Literal
from sentence_transformers import CrossEncoder
from fuzzywuzzy import process, fuzz
import re
import os

groq_api_key = os.getenv('GROQ_API_KEY')

QUERY_TEMPLATES = {
    "leaf_nodes":     "MATCH (r:Repo {repo_id: $repo})-[:CONTAINS]->(f:File) WHERE NOT (f)-[:IMPORTS]->() RETURN f.path, null as imported_path",
    "direct_imports": "MATCH (r:Repo {repo_id: $repo})-[:CONTAINS]->(f:File {name: $name}) OPTIONAL MATCH (f)-[:IMPORTS]->(i) RETURN f.path, i.path as imported_path",
    "most_deps":      "MATCH (r:Repo {repo_id: $repo})-[:CONTAINS]->(f:File) OPTIONAL MATCH (f)-[:IMPORTS]->(i) RETURN f.path, COUNT(i) as dependencies_count, collect(i.path) as imported_paths ORDER BY dependencies_count DESC LIMIT 1",
    "indirect_deps":  "MATCH (r:Repo {repo_id: $repo})-[:CONTAINS]->(f:File {name: $source})-[:IMPORTS]->(mid:File {name: $via}) OPTIONAL MATCH (mid)-[:IMPORTS]->(i) RETURN f.path, mid.path as via_path, i.path as imported_path",
    "all_files":      "MATCH (r:Repo {repo_id: $repo})-[:CONTAINS]->(f:File) OPTIONAL MATCH (f)-[:IMPORTS]->(i) RETURN f.path, i.path as imported_path",
}

class QueryTemplate(BaseModel):
    template_name: Literal['leaf_nodes', 'direct_imports', 'most_deps', 'indirect_deps', 'all_files']
    params: dict = Field(default_factory=dict, description="Parameters to fill: name, source, via")

VALID_FILE_PROPERTIES = {'name', 'path', 'functions', 'classes', 'imports'}

class CypherQuery(BaseModel):
    cypher: str = Field(..., description="Cypher query for retrieving relationships within the nodes")

class RouterDecision(BaseModel):
    decision: Literal['hybrid', 'graph_only'] = Field(..., description="graph_only ONLY for pure structural/dependency/topology questions. hybrid for everything else.")
    reasoning: str = Field(..., description="Brief reasoning behind the routing decision")
    confidence_score: float = Field(..., description="Confidence score of the decision")

class ResponseModel(BaseModel):
    answer: str = Field(..., description="Response of the query from the llm model")
    score: float = Field(..., description="Confidence score of the response")
    sources: str = Field(..., description="File name used for response")


class QueryEngine:
    def __init__(self, repo_id: str, db_client, uri: str, user: str, password: str, llm: ChatGroq):
        self.db_client = db_client
        self.repo_id = repo_id
        self.llm = llm
        self.graph_driver = GraphDatabase.driver(uri, auth=(user, password))
        self.rerank_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
        self._file_index = None

    def _load_file_index(self) -> list[dict]:
        """
        Fetch all files with their classes and functions from Neo4j.
        Cached after first call so we don't hit the DB on every query.
        """
        if self._file_index is not None:
            return self._file_index

        with self.graph_driver.session() as session:
            result = session.run(f"""
                MATCH (r:Repo {{repo_id: '{self.repo_id}'}})-[:CONTAINS]->(f:File)
                RETURN f.name as name, f.path as path,
                       f.classes as classes, f.functions as functions
            """)
            self._file_index = [dict(r) for r in result]

        return self._file_index

    def resolve_filename(self, raw_name: str) -> str | None:
        """
        Fuzzy-match a raw name (possibly missing extension or slightly misspelled)
        against actual filenames in the graph. Returns the matched name or None.
        """
        index = self._load_file_index()
        names = [f["name"] for f in index]

        if raw_name in names:
            return raw_name

        match = process.extractOne(raw_name, names, scorer=fuzz.token_sort_ratio)
        if match and match[1] >= 70:
            return match[0]

        return None

    def extract_entities_from_query(self, query: str) -> list[str]:
        """
        Pull class/function/file mentions from the query text and map them
        to file paths via fuzzy matching. Used when graph returns no filenames.
        """
        index = self._load_file_index()

        entity_map: dict[str, str] = {}
        for f in index:
            entity_map[f["name"]] = f["path"]
            for cls in (f.get("classes") or []):
                entity_map[cls] = f["path"]
            for fn in (f.get("functions") or []):
                entity_map[fn] = f["path"]

        if not entity_map:
            return []

        words = query.replace("'s", "").split()
        bigrams = [f"{words[i]} {words[i+1]}" for i in range(len(words) - 1)]
        candidates = words + bigrams

        matched_paths = []
        for candidate in candidates:
            match = process.extractOne(
                candidate,
                list(entity_map.keys()),
                scorer=fuzz.token_sort_ratio
            )
            if match and match[1] >= 75:
                matched_paths.append(entity_map[match[0]])

        return list(set(matched_paths))

    def sanitize_cypher(self, cypher: str) -> str:
        used_props = set(re.findall(r'f\.(\w+)', cypher))
        invalid = used_props - VALID_FILE_PROPERTIES
        if invalid:
            raise ValueError(f"Cypher uses invalid properties: {invalid}")
        return cypher

    def resolve_names_in_cypher(self, cypher: str) -> str:
        """
        Find name/source/via string literals in generated Cypher and
        fuzzy-resolve them to actual filenames in the graph.
        """
        def replacer(match):
            raw = match.group(1)
            resolved = self.resolve_filename(raw)
            if resolved and resolved != raw:
                print(f"[resolve] '{raw}' → '{resolved}'")
                return match.group(0).replace(raw, resolved)
            return match.group(0)

        pattern = r"\{(?:name|source|via):\s*['\"]([^'\"]+)['\"]\}"
        return re.sub(pattern, replacer, cypher)

    def _is_meaningful(self, graph_data: list[dict]) -> bool:
        """
        A graph result is meaningful only if at least one record has a
        non-null value beyond f.path — i.e. an actual relationship was found.
        Pure LIMIT 1 fallback results only have f.path + null imported_path.
        """
        return any(
            any(v is not None for k, v in record.items() if k != 'f.path')
            for record in graph_data
        )

    def graph_search(self, query: str) -> list[dict]:
        structured_llm = self.llm.with_structured_output(CypherQuery)

        system_content = f"""
        You are a Cypher query expert for Neo4j. Generate ONLY valid Cypher queries.

        SCHEMA:
        - Repo node with property repo_id
        - (Repo)-[:CONTAINS]->(File)
        - (File)-[:IMPORTS]->(File)

        FILE PROPERTIES (the ONLY valid properties):
        - name: string (just filename e.g. app.py)
        - path: string (full path e.g. src/services/tutor_service.py)
        - functions: list of strings
        - classes: list of strings
        - imports: list of strings

        CRITICAL RULES:
        1. Always start with: MATCH (r:Repo {{repo_id: '{self.repo_id}'}})
        2. Return f.path and imported.path by default. For multi-hop queries also
           return intermediate nodes as via_path or as dependency_chain list.
        3. Use OPTIONAL MATCH for relationships that may not exist.
        4. NEVER use: Type(), labels(), IS EMPTY, IS NOT EMPTY
        5. NEVER invent properties — only use name, path, functions, classes, imports
        6. For list contents use: 'value' IN f.functions or 'value' IN f.classes
        7. Only valid WHERE operators: =, <>, IN, CONTAINS, STARTS WITH, IS NULL, IS NOT NULL
        8. For indirect/transitive dependencies use: [:IMPORTS*1..5]
        9. For targeted queries (specific file or function), always add LIMIT 20.
        10. NEVER return literal strings or explanatory text. Always return node properties.
        11. If the question cannot be answered from graph structure alone, return ONLY:
            MATCH (r:Repo {{repo_id: '{self.repo_id}'}})-[:CONTAINS]->(f:File)
            RETURN f.path, null as imported_path LIMIT 1

        EXAMPLES:

        Get file and its direct imports:
        MATCH (r:Repo {{repo_id: '{self.repo_id}'}})-[:CONTAINS]->(f:File {{name: 'app.py'}})
        OPTIONAL MATCH (f)-[:IMPORTS]->(imported)
        RETURN f.path, imported.path LIMIT 20

        Find files with no imports (leaf nodes):
        MATCH (r:Repo {{repo_id: '{self.repo_id}'}})-[:CONTAINS]->(f:File)
        WHERE NOT (f)-[:IMPORTS]->()
        RETURN f.path, null as imported_path

        File with most dependencies:
        MATCH (r:Repo {{repo_id: '{self.repo_id}'}})-[:CONTAINS]->(f:File)
        OPTIONAL MATCH (f)-[:IMPORTS]->(imported:File)
        RETURN f.path, COUNT(imported) as dependencies_count, collect(imported.path) as imported_paths
        ORDER BY dependencies_count DESC LIMIT 1

        All transitive dependencies up to 5 hops:
        MATCH (r:Repo {{repo_id: '{self.repo_id}'}})-[:CONTAINS]->(f:File {{name: 'app.py'}})
        OPTIONAL MATCH path = (f)-[:IMPORTS*1..5]->(imported:File)
        RETURN f.path, [n IN nodes(path) | n.path] as dependency_chain, imported.path LIMIT 20

        What depends on a file (reverse lookup):
        MATCH (r:Repo {{repo_id: '{self.repo_id}'}})-[:CONTAINS]->(f:File {{name: 'tutor_service.py'}})
        MATCH (upstream:File)-[:IMPORTS*1..5]->(f)
        RETURN DISTINCT upstream.path, f.path LIMIT 20
        """

        response = structured_llm.invoke([
            SystemMessage(content=system_content),
            HumanMessage(content=f"Question: {query}")
        ])

        try:
            safe_cypher = self.sanitize_cypher(response.cypher)
            safe_cypher = self.resolve_names_in_cypher(safe_cypher)  # fix wrong/missing filenames

            with self.graph_driver.session() as session:
                result = session.run(safe_cypher)
                output = [record.data() for record in result]

            return output

        except ValueError as e:
            print(f"Sanitization error: {e}")
            return []
        except Exception as e:
            print(f"Cypher error: {e}")
            print(f"Failed query: {response.cypher}")
            return []

    def vector_search(self, query: str, filenames: list[str]):
        return self.db_client.collection.query(
            query_texts=[query],
            n_results=10,
            where={"path": {"$in": filenames}}
        )

    def rerank(self, context: dict, query: str, top_k: int):
        documents = context['documents'][0]
        metadatas = context['metadatas'][0]

        chunks = [(query, chunk) for chunk in documents]
        scores = self.rerank_model.predict(chunks)

        return sorted(
            [[metadatas[i], score, documents[i]] for i, score in enumerate(scores)],
            key=lambda x: x[1],
            reverse=True
        )[:top_k]

    def _extract_filenames(self, graph_data: list[dict]) -> list[str]:
        """Pull every non-null file path out of a graph result list."""
        paths = []
        for record in graph_data:
            for k, v in record.items():
                if v is None:
                    continue
                if isinstance(v, str) and k in ('f.path', 'imported.path', 'via_path'):
                    paths.append(v)
                elif k == 'dependency_chain' and isinstance(v, list):
                    paths.extend(p for p in v if p is not None)
        return list(set(paths))

    def _resolve_filenames(self, graph_data: list[dict], query: str) -> list[str]:
        """
        Try to get filenames from graph data first.
        If graph result is empty or only has the LIMIT 1 fallback garbage,
        fall back to entity extraction from the query text itself.
        """

        if graph_data and self._is_meaningful(graph_data):
            filenames = self._extract_filenames(graph_data)
            if filenames:
                return filenames

        print("[fallback] Graph returned no meaningful data, using entity extraction")
        return self.extract_entities_from_query(query)

    def router(self, query: str) -> RouterDecision:
        router_llm = self.llm.with_structured_output(RouterDecision)
        router_prompt = ChatPromptTemplate.from_messages([
            ('system', """
                You are a query router for a code repository assistant.
                Classify the query into one of two types:

                - graph_only: ONLY for pure structural questions about file
                  relationships, imports, or dependencies with NO behavioral
                  component whatsoever.
                  Examples:
                    "which files import X"
                    "what does app.py depend on"
                    "which files have no imports"
                    "list all files tutor_service imports"

                - hybrid: for EVERYTHING else — behavior, logic, implementation,
                  "what happens when", "how does X work", "why does Y do Z",
                  "what is the initial value of", "what does function X do",
                  "what checkpointing does X use", "what database does X use",
                  "what condition determines whether", "how is X constructed".
                  When in doubt, ALWAYS choose hybrid.

                KEY RULE: if the question asks about what code DOES, HOW it
                behaves, or WHAT VALUE something holds — always hybrid,
                even if it mentions a specific file or class name.
            """),
            ('user', "query: {query}")
        ])
        return (router_prompt | router_llm).invoke({"query": query})

    def get_result(self, query: str, top_k: int):
        decision  = self.router(query)
        route     = decision.decision
        reasoning = decision.reasoning
        score     = decision.confidence_score

        if route == "graph_only":
            graph_data = self.graph_search(query)

            if graph_data and self._is_meaningful(graph_data):
                return {
                    "type": "graph",
                    "graph": graph_data,
                    "confidence": score,
                    "reasoning": reasoning
                }

            filenames = self.extract_entities_from_query(query)
            if filenames:
                vector_data = self.vector_search(query, filenames=filenames)
            else:
                vector_data = self.db_client.collection.query(
                    query_texts=[query], n_results=top_k
                )

            return {
                "type": "hybrid",
                "graph": [],
                "vector": self.rerank(vector_data, query, top_k),
                "confidence": score,
                "reasoning": reasoning
            }

        else:
            graph_data = self.graph_search(query)
            filenames  = self._resolve_filenames(graph_data, query)

            if filenames:
                vector_data = self.vector_search(query, filenames=filenames)
            else:
                vector_data = self.db_client.collection.query(
                    query_texts=[query], n_results=top_k
                )

            return {
                "type": "hybrid",
                "graph": graph_data,
                "vector": self.rerank(vector_data, query, top_k),
                "confidence": score,
                "reasoning": reasoning
            }

    def close(self):
        self.graph_driver.close()

In [ ]:
from langchain_openai import ChatOpenAI

openai_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

engine = QueryEngine(repo_id=filename, db_client=client, uri="bolt://localhost:7687", user="neo4j", password="pk142145", llm=openai_llm)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2480.89it/s]


In [19]:
def formate_documents(data) -> str:
    context = ""

    if data['type'] in ('graph', 'hybrid'):
        graph_data = data['graph']
        context += f"Relationships:\n"
        for r in graph_data:
            context += f" {r.get('f.path', '')} -> {r.get('imported.path', '')} \n"
        
    if data['type'] == 'hybrid':
        context += "CHUNK DATA:\n"

        for vector_data in data['vector']:
            docs = vector_data[2]
            metas = vector_data[0]
            context += f"\n[{metas['path']} | lines {metas['line_start']}-{metas['line_end']}]\n{docs}\n"

    return context
        

In [21]:
query = "How the _render_teach_lesson function is working?"
result = engine.get_result(query, 6)
formated_result = formate_documents(result)

[fallback] Graph returned no meaningful data, using entity extraction


In [23]:
formated_result

'Relationships:\nCHUNK DATA:\n\n[src/services/tutor_service.py | lines 109-179]\n            goto="human",\n            update={\n                "phase": "teach",\n                "final_response": final_response,\n                "current_question": PROCEED_PROMPT,\n                "student_answer": "",\n            },\n        )\n\n    def _render_teach_lesson(self, concept_name: str) -> str:\n        """Build the lesson text (including equations if present)."""\n        diagnoser = DiagnoserService(self.neo4j_conn)\n        meta = diagnoser.fetch_concept_metadata(concept_name)\n        equations_block, has_equations = format_equations_for_prompt(meta.get("equations"))\n\n        equation_rules = ""\n        if has_equations:\n            equation_rules = """s\n            - The knowledge base includes **equations** for this topic. You MUST include a section titled **### Key equations** (or similar).\n            - For **each** equation listed below: state its **name**, write the **

In [24]:
from pydantic import BaseModel, Field

class ResponseModel(BaseModel):
    answer: str = Field(..., description="Response of the query from the llm model")
    score: float = Field(..., description="Confidence score of the response")
    sources: str = Field(..., description="File name used for response")

class AnswerEngine:
    def __init__(self, llm):
        self.llm = llm.with_structured_output(ResponseModel)

    def generate_response(self, query: str, context: str):
        prompt = ChatPromptTemplate.from_messages([
            ("system", """
                You are a code repository assistant.
                Answer only from the provided context.
                When asked about initial or default values, prioritize code that
                constructs or initializes objects (e.g. GraphState(...), __init__,
                initial_state) over code that updates or transitions values.
                Sources should be file names used to answer.
                Confidence should be between 0 and 1.
            """),
            ("user", "Context:\n{context}\n\nQuestion: {query}")
        ])
        chain = prompt | self.llm
        return chain.invoke({"context": context, "query": query})

In [25]:
from langchain_openai import ChatOpenAI

openai_llm = ChatOpenAI(model="gpt-4o", temperature=0)

# query = "what if i delete concept.py"
chat_engine = AnswerEngine(openai_llm)
result = chat_engine.generate_response(query, formated_result)

In [26]:
result.answer

"The `_render_teach_lesson` function is responsible for constructing a lesson text for a given concept. It uses metadata about the concept, including any equations, to build a structured lesson. Here's how it works:\n\n1. **Fetch Metadata**: It uses the `DiagnoserService` to fetch metadata for the given `concept_name`. This metadata includes information like equations, chunk type, difficulty score, description, and subtopics.\n\n2. **Format Equations**: If the metadata includes equations, it formats them for inclusion in the lesson using the `format_equations_for_prompt` function.\n\n3. **Build System Text**: It constructs a `system_text` string that outlines the rules for teaching the lesson. This includes instructions on how to present equations if they are present.\n\n4. **Build Human Text**: It constructs a `human_text` string that includes the concept name, chunk type, difficulty, description, and subtopics. If there are equations, they are also included in this text.\n\n5. **Crea

## RAG Evaluation metrices

In [27]:
eval_questions = [

    # --- EDGE: FILES WITH NO IMPORTS ---
    {
        "question": "Which files in the repository do not import any other file?",
        "ground_truth": "Files with no imports are leaf nodes in the dependency graph — likely src/models/concept.py, src/models/state.py, src/models/diagnoses.py, and src/models/intent.py as they are pure model definitions"
    },

    # --- EDGE: CIRCULAR / DEEP DEPENDENCY ---
    {
        "question": "Which service has the most dependencies on other files?",
        "ground_truth": "src/services/tutor_service.py has the most dependencies as it imports Neo4jConn, StudentDB, DiagnoserService, IntentService, PlannerService, and tutor_lesson_utils"
    },

    # --- EDGE: SPECIFIC FUNCTION BEHAVIOR ---
    {
        "question": "What happens if planned_paths is empty in the planner node?",
        "ground_truth": "If planned_paths is empty, the planner node returns Command with goto=END and a final_response message explaining either the topic was not recognized or no learning path was found"
    },

    # --- EDGE: SPECIFIC CLASS METHOD ---
    {
        "question": "What does update_last_json_path do in StudentDB and when is it called?",
        "ground_truth": "update_last_json_path updates the last_json_path field and last_updated timestamp in TinyDB for the student. It is called in app.py after successfully building the knowledge graph from a PDF"
    },

    # --- EDGE: STATE FIELD USAGE ---
    {
        "question": "What is the initial value of phase in GraphState when a new session starts?",
        "ground_truth": "The initial value of phase is 'quiz' as set in the initial_state construction in app.py"
    },

    # --- EDGE: ERROR PATH ---
    {
        "question": "What does app.py do if no concepts are extracted from the uploaded PDF?",
        "ground_truth": "app.py updates status to error state, displays an error message saying no relevant concepts were identified and to try a more detailed document, then calls st.stop()"
    },

    # --- EDGE: MULTI-HOP DEPENDENCY ---
    {
        "question": "Which files does app.py indirectly depend on through tutor_service.py?",
        "ground_truth": "Through tutor_service.py, app.py indirectly depends on neo4j_conn.py, student_db.py, diagnoser_service.py, intent_service.py, planner_service.py, and tutor_lesson_utils.py"
    },

    # --- EDGE: SPECIFIC PARAMETER ---
    {
        "question": "What is the process_limit in app.py and what does it control?",
        "ground_truth": "process_limit is set to 50 in app.py and controls the maximum number of PDF chunks processed for concept extraction to avoid excessive API calls"
    },

    # --- EDGE: TWO LLM INSTANCES ---
    {
        "question": "Why does TutorWorkflow initialize two separate LLM instances?",
        "ground_truth": "TutorWorkflow uses self.llm with max_tokens=250 for quick responses like quiz questions and self.teach_llm with max_tokens=1000 and temperature=0.35 for longer concept explanations requiring more detail"
    },

    # --- EDGE: CHECKPOINT ---
    {
        "question": "What checkpointing mechanism does TutorWorkflow use and why?",
        "ground_truth": "TutorWorkflow uses MemorySaver from langgraph.checkpoint.memory as the checkpointer when compiling the workflow, which enables conversation state persistence across multiple turns using thread_id"
    },

    # --- EDGE: THREAD ID ---
    {
        "question": "How is thread_id constructed in app.py and what is it used for?",
        "ground_truth": "thread_id is constructed as f'thread_{student_id}' in app.py and is used as the LangGraph config key to isolate each student's conversation state in MemorySaver"
    },

    # --- EDGE: RESUME VS FRESH ---
    {
        "question": "What condition determines whether the tutor resumes an existing session or starts fresh?",
        "ground_truth": "If state.values and state.next both exist, the tutor resumes via Command(resume=user_input). If either is falsy, it starts fresh by invoking with a new GraphState"
    },

    # --- EDGE: TEMP FILE CLEANUP ---
    {
        "question": "How does app.py handle the temporary PDF file after processing?",
        "ground_truth": "After processing, app.py checks if the temp file exists using os.path.exists and removes it using os.remove to clean up disk space"
    },

    # --- EDGE: MISSING INPUT GUARD ---
    {
        "question": "How does app.py handle the case where user types the placeholder text 'I want to learn about...' literally?",
        "ground_truth": "app.py checks if user_input.strip().lower() equals 'i want to learn about...' or 'i want to learn about' and replaces user_input with an empty string to avoid sending the placeholder as actual input"
    },

    # --- EDGE: DATABASE LAYER ---
    {
        "question": "What database does StudentDB use internally and what file does it store data in?",
        "ground_truth": "StudentDB uses TinyDB internally and stores data in a JSON file at the path db_folder/student_sessions.json where db_folder is determined by the student_id"
    }
]

In [28]:
results = []

for sample in eval_questions:
    raw_result = engine.get_result(sample["question"], top_k=5)
    
    context = formate_documents(raw_result)
    
    response = chat_engine.generate_response(sample["question"], context)
    
    chunks = []

    if raw_result["type"] in ("graph", "hybrid"):
        chunks.append(str(raw_result["graph"]))

    if raw_result["type"] == "hybrid":
        chunks.extend(data[2] for data in raw_result["vector"])

    results.append({
        "question": sample["question"],
        "answer": response.answer,
        "contexts": chunks,
        "ground_truth": sample["ground_truth"]
    })

Sanitization error: Cypher uses invalid properties: {'planned_paths'}
[fallback] Graph returned no meaningful data, using entity extraction
[fallback] Graph returned no meaningful data, using entity extraction
[fallback] Graph returned no meaningful data, using entity extraction
[fallback] Graph returned no meaningful data, using entity extraction
[fallback] Graph returned no meaningful data, using entity extraction
[fallback] Graph returned no meaningful data, using entity extraction
[fallback] Graph returned no meaningful data, using entity extraction
[fallback] Graph returned no meaningful data, using entity extraction
[fallback] Graph returned no meaningful data, using entity extraction
[fallback] Graph returned no meaningful data, using entity extraction
[fallback] Graph returned no meaningful data, using entity extraction


In [29]:
results

[{'question': 'Which files in the repository do not import any other file?',
  'answer': 'The `README.md` file does not import any other file.',
  'contexts': ['[]',
   'import json\nimport os\nfrom sentence_transformers import SentenceTransformer\nfrom fuzzywuzzy import process, fuzz\nfrom src.models.concept import Concept\nfrom src.database.neo4j_conn import Neo4jConn\n\nclass GraphService:\n    def __init__(self, driver_conn: Neo4jConn):\n        self.driver = driver_conn.connect()\n        # Initialize embedding model lazily if needed\n        self.embedding_model = SentenceTransformer(\'all-MiniLM-L6-v2\')\n\n    def load_concepts_from_json(self, json_path: str) -> list[Concept]:\n        if not os.path.exists(json_path):\n            raise FileNotFoundError(f"JSON file not found: {json_path}")\n        \n        with open(json_path, "r") as f:\n            data = json.load(f)\n        \n        # Exact reload logic\n        concepts = []\n        for item in data:\n            co

In [30]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    answer_correctness,
    context_precision,
    context_recall
)
from ragas.llms import LangchainLLMWrapper
from datasets import Dataset

ragas_llm = LangchainLLMWrapper(ChatOpenAI(
    model="gpt-4o-mini"
))

dataset = Dataset.from_list(results)

scores = evaluate(
    dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        answer_correctness,
        context_precision,
        context_recall
    ],
    llm=ragas_llm
)

print(scores)

Evaluating: 100%|██████████| 75/75 [00:42<00:00,  1.76it/s]

{'faithfulness': 0.7967, 'answer_relevancy': 0.6855, 'answer_correctness': 0.6416, 'context_precision': 0.4477, 'context_recall': 0.5778}


In [71]:
from typing import TypedDict

class QueryState(TypedDict):
    question: str
    vector_context: str
    graph_context: str
    agent: str
    answer: str
